<a href="https://colab.research.google.com/github/ishanallasanagala/ds2002-fa26/blob/main/notebooks/03-pandas-cleaning/2026-09-25%20%E2%80%94%20Cleaning%20Gauntlet%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [133]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [134]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])  # inject dupes
df.head()

,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


### TODO 1 — drop duplicates

In [135]:
# TODO
removed = df.duplicated().sum()
df = df.drop_duplicates().copy()

log('drop duplicates', 'dropped exact duplicate rows', removed)

[drop duplicates] dropped exact duplicate rows (15 row(s))


### TODO 2 — clean `price` -> float

In [136]:
# TODO
df['price'] = df['price'].str.strip('$').astype(float)

assert df['price'].dtype == float
log('price', 'stripped string formatting and dollar signs and converted to float', len(df))

[price] stripped string formatting and dollar signs and converted to float (300 row(s))


### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [137]:
# TODO: apply your decision, then log both quantities dropped
df['qty'] = pd.to_numeric(df['qty'], errors='coerce')

missing = df['qty'].isna().sum() # TODO: count of NaN quantities
negative = (df['qty'] < 0).sum() # TODO: count of negative quantities

df = df[df['qty'].notna() & (df['qty'] > 0)].copy()
df['qty'] = df['qty'].astype(int)

log('qty', 'dropped missing and negative quantities', missing + negative)

[qty] dropped missing and negative quantities (25 row(s))


### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [138]:
# TODO: inspect the variants, build a mapping dict, apply it, log the collapse
print(df['item'].value_counts())

df['item'] = df['item'].str.strip().str.title()
ITEM_MAP = {
    'Cheese Burger': 'Cheeseburger',
}
df['item'] = df['item'].replace(ITEM_MAP)

log('item', 'standardized spacing and casing and matched spellings to match', len(df))

item
Foam Finger      57
Rain Poncho      49
cheese burger    44
Cheeseburger     43
rain poncho      42
foam finger      40
Name: count, dtype: int64
[item] standardized spacing and casing and matched spellings to match (275 row(s))


### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [139]:
# TODO
print(df['category'].value_counts())

df['category'] = (
    df['category'].str.strip()
    .str.lower()
    .str.replace('-','',regex=False)
)

#For the business decision part of combining the values for Apparel and Merch
CATEGORY_MAP = {
    'apparel': 'merch'
}
df['category'] = df['category'].replace(CATEGORY_MAP)

log('category', 'Combined the values within the categories of Apparel and Merch', len(df))

category
Food         51
Merch        51
rain-gear    45
food         44
Apparel      43
RainGear     41
Name: count, dtype: int64
[category] Combined the values within the categories of Apparel and Merch (275 row(s))


### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [140]:
assert df.duplicated().sum() == 0
assert df['qty'].min() >= 1
assert df['price'].dtype == float
# TODO: assert something about item
# 4. Item has no nulls
assert df['item'].isna().sum() == 0, 'missing items remain'
# TODO: assert something about category
# 5. Categories are all lowercase
assert df['category'].str.islower().all(), 'inconsistent category text'
print('clean:', df.shape)

clean: (275, 5)


### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [141]:
# TODO
df['revenue'] = df['qty'] * df['price']

#Print revenue by category, ordered from highest to lowest
print(df.groupby('category')['revenue'].sum().sort_values(ascending=False).round(2))

print(f"\nOverall Revenue Total ${df['revenue'].sum():.2f}")

category
food        1656.0
merch       1572.0
raingear    1512.0
Name: revenue, dtype: float64

Overall Revenue Total $4740.00


**What I would tell the vendor:** I would tell the vendor to continue to prioritize food, as it is the highest revenue-producing category. I would also tell the vendor do not neglect the well-roundedness of the categories sold, as they are all fairly effective categories for selling.

### TODO 8 — read back your log

In [142]:
import pandas as pd
pd.DataFrame(DECISIONS)

,step,decision,rows
0,drop duplicates,dropped exact duplicate rows,15
1,price,stripped string formatting and dollar signs an...,300
2,qty,dropped missing and negative quantities,25
3,item,standardized spacing and casing and matched sp...,275
4,category,Combined the values within the categories of A...,275


### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.

Step 3 changed our revenue total the most, as it dropped the negative quantities present within the system. The revenue total before the cleaning step would have been 4594.50 dollars. The revenue total after that step comes out to be 4740.00 dollars.

**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.

I decided to combine the categories of Merch and Apparel into one, given the similarities between the two in producing branded consumer goods supplied similarly. Given the minor differences between the two, with Merch typically encompassing Apparel, a reasonable person could have maintained their distinct presence. This wouldn't change the Overall Total Revenue, but it would split the revenue within Merch between Merch and apparel when the categories were reported in descending order of total revenue.